# Chapter 05 · Notebook 01: Our Model as a PyTorch Layer

**Math companion:** [Beginner mathematical view](00_math_intuition.ipynb). Study its relevant sections alongside these lessons.

[Course index](../README.md) · [Chapter guide](README.md) ·
[Previous chapter](../04_optimizers_and_momentum/README.md)

**Goal:** express our familiar `prediction = weight × input + bias` calculation as
`torch.nn.Linear(1, 1)`. Inspect the two shared parameters and the data shape before training.

Run this notebook from top to bottom in a fresh kernel. It contains its own imports
and data. Write predictions before running each lesson. Training the layer is the next
notebook; this one only constructs, inspects, initializes, and evaluates its calculations.

## Contents

- [Setup](#setup)
- [01. Create a layer and inspect its parameters](#lesson-01)
- [02. Arrange examples and features](#lesson-02)
- [03. Make predictions with familiar parameters](#lesson-03)
- [04. Measure the same mean squared loss](#lesson-04)


## Setup

<a id="setup"></a>

Import PyTorch. We will use the existing installation; no package changes are needed.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)


## Lesson 01: Create a layer and inspect its parameters

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#matrix-shapes).

<a id="lesson-01"></a>

`torch.nn.Linear(1, 1)` accepts one input feature per example and produces one output
value per example. It performs our familiar weight-times-input-plus-bias calculation.

The layer owns its parameters as `model.weight` and `model.bias`. Both are learnable
by default, but creating the layer does not train it. The layer initially chooses
random parameter values; we will assign familiar starting values in lesson 03.

Weight has shape `[1, 1]`: one output connected to one input. Bias has shape `[1]`:
one bias for the output. Each tensor contains one number, so there are **two trainable
numbers in total**, shared across all examples. `.numel()` counts numbers in a tensor.

**Before running:** predict the total parameter count. Does using four training examples
mean the layer needs four sets of weights and biases?

**My prediction:** _Write here._

**Observation and explanation:** _Write here._


In [ ]:
# One input feature and one output value; bias is included by default.
model = torch.nn.Linear(in_features=1, out_features=1)

print("Layer:", model)
print("Weight shape:", model.weight.shape)
print("Bias shape:", model.bias.shape)
print("Weight requires gradients:", model.weight.requires_grad)
print("Bias requires gradients:", model.bias.requires_grad)

# Count individual learned numbers, rather than the number of training examples.
parameter_count = model.weight.numel() + model.bias.numel()
print("Total parameter values:", parameter_count)


## Lesson 02: Arrange examples and features

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#matrix-shapes).

<a id="lesson-02"></a>

Our old input tensor had shape `[4]`. For a batch passed to this layer, arrange it as
`[number of examples, number of input features]`, which is `[4, 1]` here:

```text
[[0],   ← example 1, one feature
 [1],   ← example 2, one feature
 [2],   ← example 3, one feature
 [3]]   ← example 4, one feature
```

`.reshape(4, 1)` arranges the same four values into four rows and one column. It does
not change their values. Targets also have shape `[4, 1]`, matching one output per row
so each prediction is compared with its corresponding target.

**Before running:** what do the `4` and `1` mean in `x.shape`? Are four examples the same
thing as four input features?

**My prediction:** _Write here._

**Observation and explanation:** _Write here._


In [ ]:
# The values and input-target pairing are unchanged from earlier chapters.
flat_x = torch.tensor([0., 1., 2., 3.])
flat_y = torch.tensor([2., 5., 8., 11.])

# Four examples, each containing one input feature and one target value.
x = flat_x.reshape(4, 1)
y = flat_y.reshape(4, 1)

print("Old input shape:", flat_x.shape)
print("Layer input shape:", x.shape)
print("Target shape:", y.shape)
print("Inputs as rows:\n", x)


## Lesson 03: Make predictions with familiar parameters

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#weighted-sum).

<a id="lesson-03"></a>

Assign weight 1 and bias 0 to reproduce our earlier starting model. This is explicit
initialization, not learning from data. `.fill_()` changes the existing parameter's
values. `torch.no_grad()` avoids tracking that initialization for gradients.

Calling `model(x)` performs the forward calculation for all four examples. It does
not apply a training update. Compare it with `1 * x + 0`.

**Before running:** predict the four outputs and the output shape. Does calling the
model change its parameters?

**My prediction:** _Write here._

**Observation and explanation:** _Write here._


In [ ]:
# Initialize the existing layer parameters to our familiar starting values.
with torch.no_grad():
    model.weight.fill_(1.)
    model.bias.fill_(0.)

# A forward pass applies the same learned parameters to every row.
predictions = model(x)
manual_predictions = 1. * x + 0.

print("Weight:", model.weight.item())
print("Bias:", model.bias.item())
print("Output shape:", predictions.shape)
# Flatten only for a compact display; predictions itself still has shape [4, 1].
print("Predictions:", predictions.detach().flatten().tolist())
print("Matches manual calculation:", torch.allclose(predictions, manual_predictions))


## Lesson 04: Measure the same mean squared loss

**Mathematical view:** [Related explanation and worked example](../01_training_foundations/00_math_intuition.ipynb#loss).

<a id="lesson-04"></a>

The model is now represented by a PyTorch layer, but our loss calculation stays the
same. Subtract matching targets, square the errors, and average them.
No `backward()` or optimizer update has happened in this notebook.

**Before running:** should this loss match the original model's loss of 30 at weight
1 and bias 0? Explain why.

**My prediction:** _Write here._

**Observation and explanation:** _Write here._


In [ ]:
# Predictions and targets both have shape [4, 1].
squared_errors = (predictions - y) ** 2
loss = squared_errors.mean()

print("Squared errors:", squared_errors.detach().flatten().tolist())
print("Mean squared loss:", loss.item())
print("Parameters still:", model.weight.item(), model.bias.item())


## What this lesson establishes

- A `Linear(1, 1)` layer expresses the same calculation as our scalar weight and bias.
- The layer holds one weight and one bias; examples share those parameters.
- A batch of four one-feature examples has shape `[4, 1]`.
- `model(x)` makes predictions; it does not update the parameters.
- Loss remains mean squared error. Training will still need gradients and optimizer updates.

**Next lesson:** give `model.parameters()` to SGD and train this layer with the loop
we already understand. We will add that notebook after reviewing this one.

API reference: [PyTorch Linear documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html).
The code was verified against the installed PyTorch version.
